<a href="https://colab.research.google.com/github/ese-ada-lovelace-2025/irp-wm722/blob/main/VetoClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import glob
import torch.nn as nn
from google.colab import drive
drive.mount('/content/drive')
import sys
from sklearn.model_selection import train_test_split
sys.path.append('/content/drive/MyDrive/IRP/scripts')
from Veto import VetoClassifier
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
import cv2
import copy
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix


In [ ]:
def set_seed(seed):
    """
    Use this to set ALL the random seeds to a fixed value and take out any randomness from cuda kernels
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False  ##uses the inbuilt cudnn auto-tuner to find the fastest convolution algorithms. -
    torch.backends.cudnn.enabled   = False

    return True

device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")

In [ ]:
paths = glob.glob('/content/drive/MyDrive/IRP/VetoTiles/*/*/*.npz')

from sklearn.model_selection import train_test_split

t, v = train_test_split(paths, test_size = 0.2)


In [ ]:
import cv2
import numpy as np
import torch

from torch.utils.data import Dataset

# AI assistance in handling shapes and channels for VetoDataset
class VetoDataset(Dataset):

    def __init__(
        self,
        paths,
        rgb_size=96,
        dem_size=300,
    ):
        self.paths = list(paths)
        self.rgb_size = rgb_size
        self.dem_size = dem_size

    def __len__(self):
        return len(self.paths)

    @staticmethod
    # make sure rgb is [h, w, band]
    def _ensure_rgb_hwc(rgb):
        rgb = np.asarray(rgb)

        if (rgb.ndim == 3 and rgb.shape[0] == 3 and rgb.shape[-1] != 3):
            rgb = np.moveaxis(rgb, 0, -1)

        return rgb

    def __getitem__(self, idx):

        path = self.paths[idx]

        with np.load(path) as data:

            rgb = data["rgb"].astype(np.float32, copy=True)

            mask = data["mask"].astype(np.float32, copy=True)

            dem_context = data["dem_context"].astype(np.float32, copy=True)

            scalar_features = data["scalar_features"].astype(np.float32, copy=True)

            label = float(data["label"])


        # local RGB + candidate-mask branch
        rgb = self._ensure_rgb_hwc(rgb)

        mask = (mask > 0.5).astype(np.float32)

        # add mask to rgb
        rgb_local = np.concatenate([rgb, mask[..., None],], axis=-1,)

        # bands to front
        rgb_local = np.moveaxis(rgb_local, -1, 0)

        # wider DEM-context branch

        if dem_context.shape[1:] != (self.dem_size, self.dem_size):
            # Resize in channels-last form.
            dem_hwc = np.moveaxis(dem_context,0,-1, )

            dem_hwc = cv2.resize(dem_hwc, (self.dem_size, self.dem_size), interpolation=cv2.INTER_LINEAR)

            # Restore the candidate-mask channel using
            # nearest-neighbour interpolation.
            context_mask = cv2.resize(dem_context[2], (self.dem_size, self.dem_size), interpolation=cv2.INTER_NEAREST)

            dem_hwc[..., 2] = (context_mask > 0.5).astype(np.float32)

            dem_context = np.moveaxis(dem_hwc, -1, 0)

        scalar_features = np.asarray(scalar_features, dtype=np.float32).reshape(-1)

        # to tensors
        rgb_local = torch.from_numpy(np.ascontiguousarray(rgb_local, dtype=np.float32))
        dem_context = torch.from_numpy(np.ascontiguousarray(dem_context, dtype=np.float32))
        scalar_features = torch.from_numpy(np.ascontiguousarray(scalar_features, dtype=np.float32))

        # 1 is accept, 0 is reject
        label = torch.tensor(label, dtype=torch.float32)

        return (rgb_local, dem_context, scalar_features, label)

In [ ]:

train_set = VetoDataset(t)
val_set = VetoDataset(v)

train_loader = DataLoader(
    train_set,
    batch_size=8,
    shuffle=True)

val_loader = DataLoader(
    val_set,
    batch_size=8)

In [ ]:
model = VetoClassifier(dropout=0.4).to(device) # drop out too high? change maybe

# get training set wide scalar stats
training_scalars = np.stack([np.load(path)["scalar_features"].astype(np.float32) for path in train_set.paths], axis=0)

scalar_mean = training_scalars.mean(axis=0)
scalar_std = np.maximum(training_scalars.std(axis=0),1e-6)
model.set_scalar_statistics(scalar_mean,scalar_std)


# params
n_epochs = 25
decision_threshold = 0.85
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-5, weight_decay=1e-4)
# Initialize the learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# label 1 = RETAIN
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(5.0, device=device))

train_losses = []
val_losses = []

f1s = []
precisions = []
recalls = []

best_val_loss = float("inf")
best_state = None
best_epoch = None


for epoch in range(n_epochs):

    # Training
    model.train()

    running_loss = 0.0
    train_samples = 0

    for rgb_local, dem_context, scalar_features, labels in train_loader:

        rgb_local = rgb_local.to(device,
                                 dtype=torch.float32,
                                 non_blocking=True)

        dem_context = dem_context.to(device,
                                     dtype=torch.float32,
                                     non_blocking=True)

        scalar_features = scalar_features.to(device,
                                             dtype=torch.float32,
                                             non_blocking=True)

        # view(-1), rather than squeeze(), remains safe
        # when the final batch contains one sample.
        labels = labels.to(device, dtype=torch.float32, non_blocking=True).view(-1)

        optimizer.zero_grad(set_to_none=True)

        logits = model(rgb_local, dem_context, scalar_features).view(-1)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.shape[0]

        running_loss += loss.item() * batch_size
        train_samples += batch_size

    epoch_train_loss = running_loss / max(train_samples, 1)
    train_losses.append(epoch_train_loss)


    # Validation
    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    all_probabilities = []
    all_labels = []

    with torch.inference_mode():

        for rgb_local, dem_context, scalar_features, labels in val_loader:

            rgb_local = rgb_local.to(device,
                                     dtype=torch.float32,
                                     non_blocking=True)

            dem_context = dem_context.to(device,
                                         dtype=torch.float32,
                                         non_blocking=True)

            scalar_features = scalar_features.to(device,
                                                 dtype=torch.float32,
                                                 non_blocking=True)

            labels = labels.to(device,
                               dtype=torch.float32,
                               non_blocking=True
                               ).view(-1)

            logits = model(rgb_local, dem_context, scalar_features).view(-1)

            loss = criterion(logits, labels)

            probabilities = torch.sigmoid(logits)

            batch_size = labels.shape[0]

            running_val_loss += loss.item() * batch_size
            val_samples += batch_size
            all_probabilities.append(probabilities.cpu())

            all_labels.append(labels.cpu())

    epoch_val_loss = (running_val_loss / max(val_samples, 1))

    val_losses.append(epoch_val_loss)

    probabilities = torch.cat(all_probabilities).numpy()

    labels_numpy = torch.cat(all_labels).numpy().astype(np.int64)

    predictions = (probabilities >= decision_threshold).astype(np.int64)

    # calculate metrics
    epoch_precision = precision_score(labels_numpy, predictions,
                                      pos_label=1, average="binary",
                                      zero_division=0)
    epoch_recall = recall_score(labels_numpy, predictions, pos_label=1,
                                average="binary", zero_division=0)

    epoch_f1 = f1_score(labels_numpy, predictions, pos_label=1,
                        average="binary", zero_division=0)

    precisions.append(epoch_precision)
    recalls.append(epoch_recall)
    f1s.append(epoch_f1)

    tn, fp, fn, tp = confusion_matrix(labels_numpy, predictions, labels=[0, 1]).ravel()

    # Step the scheduler
    scheduler.step(epoch_val_loss)

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:03d}/{n_epochs} | "
        f"train {epoch_train_loss:.4f} | "
        f"val {epoch_val_loss:.4f} | "
        f"P {epoch_precision:.3f} | "
        f"R {epoch_recall:.3f} | "
        f"F1 {epoch_f1:.3f} | "
        f"TP {tp} FP {fp} FN {fn} TN {tn}"
    )


### Confusion Matrix Visualization

Let's visualize the confusion matrix for the final epoch to better understand the model's performance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = np.array([[tn, fp], [fn, tp]])

fig = plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Reject', 'Predicted Accept'], yticklabels=['Actual Reject', 'Actual Accept'])
plt.title('Confusion Matrix for Last Epoch')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
torch.save(best_state, '/content/drive/MyDrive/IRP/models/VetoModelNew.pt')